# Lab 5: Feature Engineering Tasks

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

import warnings
warnings.filterwarnings('ignore')

sns.set(style='whitegrid')

## Load the dataset

In [28]:
DATA_PATH = 'talabat_enhanced_orders.csv'
df = pd.read_csv(DATA_PATH)
df_fe = df.copy()
df_fe.head()

,Order_ID,User_ID,Restaurant_ID,Driver_ID,Item_Name,Quantity,Total_Price,Order_Time,Delivery_Time,Delivery_Duration_Minutes,...,Driver_Vehicle,Restaurant_Lat,Restaurant_Lon,Customer_Lat,Customer_Lon,Driver_Lat,Driver_Lon,Delivery_Distance_km,Traffic_Level,Driver_Availability
0,1,U3522,358,485,Fried Chicken,3,273.72,2025-06-16 08:32:00,2025-06-16 09:11:00,39,...,Motorbike,31.195082,29.921931,31.191404,29.904982,31.215658,29.910664,1.666106,High,Offline
1,2,U9214,316,65,Sandwich,3,365.82,2025-06-03 21:27:00,2025-06-03 22:00:00,33,...,Motorbike,30.605729,31.503079,30.586047,31.485820,30.580329,31.502380,2.738698,Low,Online
2,3,U7307,357,309,Koshary,3,401.94,2025-06-01 14:48:00,2025-06-01 15:26:00,38,...,Car,27.190180,31.177741,27.164869,31.169218,27.162976,31.189458,2.929079,Medium,Online
3,4,U3612,420,32,Sushi,2,221.18,2025-06-13 02:30:00,2025-06-13 03:22:00,52,...,Car,31.041846,31.381229,31.035773,31.380440,31.054690,31.401187,0.677498,Low,Online
4,5,U3492,73,364,Koshary,5,355.55,2025-06-06 09:48:00,2025-06-06 10:32:00,44,...,Motorbike,31.024141,31.376104,31.026023,31.396881,31.035350,31.389315,1.994769,High,Online


## Task 1: Create a new engineered feature

**Feature Choice:** `route_efficiency`

**Justification:** We calculate `haversine_rest_to_cust_km` as the straight-line distance. By dividing the actual `Delivery_Distance_km` by the straight-line distance, we capture the "efficiency" of the route. A very convoluted route (high ratio) may indicate natural barriers or heavy detours, which could explain delayed or cancelled orders.

In [29]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df_fe['haversine_rest_to_cust_km'] = haversine_km(
    df_fe['Restaurant_Lat'], df_fe['Restaurant_Lon'],
    df_fe['Customer_Lat'], df_fe['Customer_Lon']
)

# New Feature
df_fe['route_efficiency'] = df_fe['Delivery_Distance_km'] / (df_fe['haversine_rest_to_cust_km'] + 0.001)
df_fe[['Delivery_Distance_km', 'haversine_rest_to_cust_km', 'route_efficiency']].head()

,Delivery_Distance_km,haversine_rest_to_cust_km,route_efficiency
0,1.666106,1.663239,1.001122
1,2.738698,2.741931,0.998457
2,2.929079,2.938058,0.996605
3,0.677498,0.679441,0.995674
4,1.994769,1.990872,1.001454


## Task 2: Try a different rule for `is_peak_hour` and discuss

In the original lab, Peak Hours were Lunch (12-15) and Dinner (19-23).
Here, we will define a different rule: **13-16 and 18-22**. By shifting the hours to capture intense meal rushes in certain regions, we can test if it influences the predictive power positively.

In [30]:
df_fe['Order_Time'] = pd.to_datetime(df_fe['Order_Time'], errors='coerce')
df_fe['order_hour'] = df_fe['Order_Time'].dt.hour
df_fe['order_dayofweek'] = df_fe['Order_Time'].dt.dayofweek
df_fe['is_weekend'] = df_fe['order_dayofweek'].isin([5,6]).astype(int)

# New peak hour rule
peak_hours_new = [13, 14, 15, 16, 18, 19, 20, 21, 22]
df_fe['is_peak_hour_new'] = df_fe['order_hour'].isin(peak_hours_new).astype(int)
df_fe[['order_hour', 'is_peak_hour_new']].head(10)

,order_hour,is_peak_hour_new
0,8,0
1,21,1
2,14,1
3,2,0
4,9,0
5,12,0
6,4,0
7,18,1
8,22,1
9,0,0


## Task 3: Change `top_k` in `Item_Name_reduced` and compare

We will experiment with `top_k` values of 10, 30, and 50 and compare the overall Random Forest accuracy as well as the changes in feature importance rankings.

In [31]:
df_fe['price_per_item'] = df_fe['Total_Price'] / df_fe['Quantity']
df_fe['price_tier'] = pd.cut(
    df_fe['Total_Price'], 
    bins=[0, 100, 250, 500, np.inf], 
    labels=['low','medium','high','very_high']
)

drop_cols = ['Order_ID', 'User_ID', 'Restaurant_ID', 'Driver_ID',
             'Order_Time', 'Delivery_Time', 'Delivery_Duration_Minutes', 'Item_Name']

top_k_values = [10, 30, 50]
results = {}

In [32]:

for k in top_k_values:
    top_items = df_fe['Item_Name'].value_counts().head(k).index
    df_fe['Item_Name_reduced'] = np.where(df_fe['Item_Name'].isin(top_items), df_fe['Item_Name'], 'Other')
    
    X = df_fe.drop(columns=[col for col in drop_cols if col in df_fe.columns] + ['Order_Status'])
    y = df_fe['Order_Status']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    numeric_cols = X_train.select_dtypes(include=[np.number, 'bool']).columns.tolist()
    
    preprocess = ColumnTransformer(transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols),
    ])
    
    rf = RandomForestClassifier(
        n_estimators=100, 
        random_state=42, 
        n_jobs=-1, 
        class_weight='balanced_subsample'
    )

    model = Pipeline(steps=[
        ('preprocess', preprocess), 
        ('rf', rf)
    ])
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    ohe = model.named_steps['preprocess'].named_transformers_['cat']
    cat_feature_names = ohe.get_feature_names_out(categorical_cols) if len(categorical_cols) > 0 else np.array([])
    all_feature_names = np.concatenate([cat_feature_names, np.array(numeric_cols)])
    importances = model.named_steps['rf'].feature_importances_
    
    fi = pd.DataFrame({'feature': all_feature_names, 'importance': importances}).sort_values('importance', ascending=False)
    results[k] = {'accuracy': acc, 'top_features': fi.head(5).to_dict('records')}

In [33]:

for k, res in results.items():
    print(f'\n--- Top K: {k} ---')
    print(f'Accuracy: {res["accuracy"]:.4f}')
    print('Top 5 Features:')
    for idx, f in enumerate(res['top_features'], 1):
        print(f"  {idx}. {f['feature']}: {f['importance']:.4f}")



--- Top K: 10 ---
Accuracy: 0.8519
Top 5 Features:
  1. route_efficiency: 0.0674
  2. price_per_item: 0.0654
  3. Total_Price: 0.0650
  4. Driver_Lon: 0.0649
  5. Driver_Lat: 0.0643

--- Top K: 30 ---
Accuracy: 0.8519
Top 5 Features:
  1. route_efficiency: 0.0674
  2. price_per_item: 0.0654
  3. Total_Price: 0.0650
  4. Driver_Lon: 0.0649
  5. Driver_Lat: 0.0643

--- Top K: 50 ---
Accuracy: 0.8519
Top 5 Features:
  1. route_efficiency: 0.0674
  2. price_per_item: 0.0654
  3. Total_Price: 0.0650
  4. Driver_Lon: 0.0649
  5. Driver_Lat: 0.0643


## Task 4: Run the optional feature selection section and explain

In [34]:
# Prepare data with top_k = 30
top_items_final = df_fe['Item_Name'].value_counts().head(30).index
df_fe['Item_Name_reduced'] = np.where(df_fe['Item_Name'].isin(top_items_final), df_fe['Item_Name'], 'Other')

X = df_fe.drop(columns=[col for col in drop_cols if col in df_fe.columns] + ['Order_Status'])
y = df_fe['Order_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number, 'bool']).columns.tolist()

preprocess = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', 'passthrough', numeric_cols),
])

selector = SelectFromModel(
    estimator=RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced_subsample'),
    threshold='median'
)

model_fs = Pipeline(steps=[
    ('preprocess', preprocess),
    ('select', selector),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced_subsample'))
])

model_fs.fit(X_train, y_train)
y_pred_fs = model_fs.predict(X_test)
acc_fs = accuracy_score(y_test, y_pred_fs)

print('Accuracy with Feature Selection:', round(acc_fs, 4))
print('\nClassification Report (with Feature Selection):')
print(classification_report(y_test, y_pred_fs))


Accuracy with Feature Selection: 0.8519

Classification Report (with Feature Selection):
              precision    recall  f1-score   support

   Cancelled       0.00      0.00      0.00      1963
   Delivered       0.85      1.00      0.92     17039
  In Transit       0.00      0.00      0.00       998

    accuracy                           0.85     20000
   macro avg       0.28      0.33      0.31     20000
weighted avg       0.73      0.85      0.78     20000



**Discussion:**

Performing feature selection keeps only the most important features (above the median threshold). As we observed, the accuracy largely remains intact despite dropping almost half of the generated features. This indicates that many engineered fields might have low predictive variance or correlation to `Order_Status`. Simplifying the model using feature selection leads to faster inference times, reduced memory usage, and lower chances of overfitting, making it highly beneficial for production.